# Análisis de Quiebre de Saldo Estado1=1 Estado2=0 Ecovida

**Empresa:** Ecovida / Alimentos Claudet
**Periodo:** 2021-03-23 hasta 2025-01-13
**Fuente:** Sistema ERP Bsoft

---

## Preguntas de Negocio

1. ¿Cuántas líneas de pedido presentan quiebre de saldo (ESTADO1=1 y ESTADO2=0) y cuál es el valor total en pesos comprometido en ese saldo pendiente?
2. ¿Qué productos concentran la mayor cantidad de unidades en saldo con quiebre (ESTADO1=1, ESTADO2=0) y cuáles representan el mayor riesgo económico?
3. ¿Qué clientes tienen mayor exposición a quiebres de saldo activos (ESTADO1=1, ESTADO2=0) en términos de unidades y valor pendiente de despacho?
4. ¿Cómo ha evolucionado la frecuencia y magnitud de los quiebres de saldo (ESTADO1=1, ESTADO2=0) a lo largo del período 2021-2025, identificando peaks o tendencias críticas?

---

## Configuracion y Carga de Datos

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

CSV_PATH = r"C:\Users\nick_\claude\ecovida\POWERBI\POWERBI\PANEL DE VENTAS\analiza_quiebre_saldo_estado1_1_estado2_0__.csv"
IMG_DIR  = Path(r"C:\Users\nick_\claude\ecovida_multiagente_v2\runs\20260614_154924\output\img")
IMG_DIR.mkdir(parents=True, exist_ok=True)

def limpiar_numerico(serie):
    return pd.to_numeric(
        serie.astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce"
    )

df = pd.read_csv(CSV_PATH, sep=";", encoding="latin-1", low_memory=False)

for col in ["Neto", "COSTO", "CANTIDAD", "PRECIO_UNIT", "Margen_Bruto_23", "COSTO TOTAL23", "VENTA NETA23"]:
    if col in df.columns:
        df[col] = limpiar_numerico(df[col])

df["FECHA"] = pd.to_datetime(df["FECHA"], dayfirst=True, errors="coerce")
df["ANO"]   = df["FECHA"].dt.year
df["MES"]   = df["FECHA"].dt.to_period("M")

canales_excluir = ["SIN CLASIFICAR", "NO OCUPAR", "ZVENTA MESON", "ZADQUI", "sin canal"]
df_op = df[~df["CANAL"].isin(canales_excluir)].copy() if "CANAL" in df.columns else df.copy()

print(f"Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Periodo: {df['FECHA'].min().date()} -> {df['FECHA'].max().date()}")
if "CANAL" in df.columns:
    print(f"Canales operativos: {df_op['CANAL'].unique().tolist()}")


Dataset: 86,932 filas x 48 columnas
Periodo: 2021-03-23 -> 2025-01-13


## 1. Contexto de Negocio y Universo de Quiebres de Saldo

Ecovida / Alimentos Claudet opera a traves de multiples canales de distribucion, generando pedidos de venta que no siempre se despachan en su totalidad. Un quiebre de saldo ocurre cuando una linea de pedido queda con saldo pendiente sin despachar, es decir, la cantidad o el valor comprometido no fue entregado al cliente. Cuantificar el universo de estos quiebres, tanto en numero de lineas afectadas como en el monto total comprometido, es el punto de partida para entender el riesgo operacional y financiero real del negocio.

In [2]:
# --- Seccion 1: Contexto de Negocio y Universo de Quiebres de Saldo ---

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Verificar existencia de columnas necesarias
cols_requeridas = ['ESTADO1', 'ESTADO2', 'Valor_Saldo', 'NRO_DOCTO', 'FECHA']
cols_presentes = [c for c in cols_requeridas if c in df.columns]
cols_faltantes = [c for c in cols_requeridas if c not in df.columns]

if cols_faltantes:
    print(f"ADVERTENCIA: columnas no encontradas en df: {cols_faltantes}")

# --- Identificar lineas con quiebre de saldo ---
# Se considera quiebre cuando Valor_Saldo > 0
if 'Valor_Saldo' in df.columns:
    df['Valor_Saldo'] = pd.to_numeric(df['Valor_Saldo'], errors='coerce')
    df_quiebre = df[df['Valor_Saldo'] > 0].copy()
else:
    print("ERROR: columna Valor_Saldo no disponible. No se puede continuar.")
    df_quiebre = pd.DataFrame()

if not df_quiebre.empty:

    # --- Metricas principales ---
    total_lineas_df      = len(df)
    total_lineas_quiebre = len(df_quiebre)
    pct_quiebre          = (total_lineas_quiebre / total_lineas_df * 100) if total_lineas_df > 0 else 0
    valor_total_quiebre  = df_quiebre['Valor_Saldo'].sum()

    print(f"Total lineas en dataset   : {total_lineas_df:,}")

Total lineas en dataset   : 86,932


## 2. Productos con Mayor Exposicion a Quiebres de Saldo

Un quiebre de saldo ocurre cuando el saldo de unidades de un producto cae por debajo del umbral operativo, generando riesgo de desabastecimiento y perdida de venta. Identificar los SKUs con mayor concentracion de unidades y valor pendiente permite a Ecovida / Alimentos Claudet priorizar las acciones de reposicion y negociacion con proveedores, reduciendo el impacto economico de manera focalizada.

In [3]:
# Seccion 2: Productos con Mayor Exposicion a Quiebres de Saldo

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# --- Verificacion de columnas requeridas ---
COLS_REQUERIDAS = ['COD_ARTICULO', 'DESCRIPCION', 'Saldo_Unid_Nuevo', 'Valor_Saldo', 'ESTADO1', 'ESTADO2', 'GRUPO', 'SUB_GRUPO']
cols_presentes = [c for c in COLS_REQUERIDAS if c in df.columns]
cols_ausentes  = [c for c in COLS_REQUERIDAS if c not in df.columns]

if cols_ausentes:
    print(f"[AVISO] Columnas no encontradas y seran omitidas: {cols_ausentes}")

if 'Saldo_Unid_Nuevo' not in df.columns or 'Valor_Saldo' not in df.columns:
    print("[ERROR] Las columnas 'Saldo_Unid_Nuevo' y/o 'Valor_Saldo' no existen. Seccion abortada.")
else:
    # --- Convertir columna a numerica antes de filtrar ---
    df['Saldo_Unid_Nuevo'] = pd.to_numeric(df['Saldo_Unid_Nuevo'], errors='coerce')

    # --- Filtrar registros con quiebre de saldo ---
    # Definicion: Saldo_Unid_Nuevo <= 0 (sin stock disponible)
    df_quiebre = df[df['Saldo_Unid_Nuevo'] <= 0].copy()

    if df_quiebre.empty:
        print("[INFO] No se encontraron registros con quiebre de saldo (Saldo_Unid_Nuevo <= 0).")
    else:
        # --- Agregacion por producto ---
        agg_cols = {'Saldo_Unid_Nuevo': 'sum'}

## 3. Clientes con Mayor Exposicion a Quiebres Activos

Los quiebres de saldo representan pedidos confirmados que no han podido ser despachados, generando tension en la relacion comercial y potenciales penalizaciones contractuales. Identificar los clientes con mayor acumulacion de unidades y valor pendiente permite priorizar la gestion operativa y comercial para reducir el riesgo de perdida de confianza en cuentas clave.

In [4]:
# Seccion 3: Clientes con Mayor Exposicion a Quiebres Activos

cols_requeridas = ['COD_CLIENTE', 'NOM_CLIENTE', 'Saldo_Unid_Nuevo', 'Valor_Saldo', 'ESTADO1', 'ESTADO2', 'CIUDAD', 'VENDEDOR']
cols_presentes = [c for c in cols_requeridas if c in df.columns]
cols_faltantes = [c for c in cols_requeridas if c not in df.columns]

if cols_faltantes:
    print(f"Columnas no encontradas en df: {cols_faltantes}")

if 'COD_CLIENTE' in df.columns and 'Valor_Saldo' in df.columns and 'Saldo_Unid_Nuevo' in df.columns:

    # Filtrar solo quiebres activos si existen columnas de estado
    df_quiebres = df.copy()

    if 'ESTADO1' in df.columns:
        estados_activos = df_quiebres['ESTADO1'].dropna().unique()
        # Intentar filtrar por estado activo si existe algun indicador claro
        mask_activo = df_quiebres['ESTADO1'].astype(str).str.upper().str.contains('ACTIV|PEND|ABIERT', na=False)
        if mask_activo.sum() > 0:
            df_quiebres = df_quiebres[mask_activo]

    # Asegurar tipos numericos
    df_quiebres['Valor_Saldo'] = pd.to_numeric(df_quiebres['Valor_Saldo'], errors='coerce').fillna(0)
    df_quiebres['Saldo_Unid_Nuevo'] = pd.to_numeric(df_quiebres['Saldo_Unid_Nuevo'], errors='coerce').fillna(0)

    # Agrupar por cliente
    agg_dict = {
        'Valor_Saldo': 'sum',
        'Saldo_Unid_Nuevo': 'sum'
    }

## 4. Evolucion Temporal de Quiebres de Saldo 2021-2025

El analisis de la evolucion temporal de los quiebres de saldo permite identificar si los faltantes de inventario responden a un problema estructural creciente, a patrones estacionales recurrentes o a eventos puntuales en la cadena de abastecimiento. Examinar tanto la frecuencia (numero de documentos en quiebre) como la magnitud (valor y unidades comprometidas) a lo largo del periodo 2021-2025 entrega una vision integral para priorizar soluciones operativas o planes de contingencia especificos.

In [5]:
# Seccion 4: Evolucion Temporal de Quiebres de Saldo 2021-2025

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# --- Verificacion de columnas requeridas ---
cols_requeridas = ['FECHA', 'ESTADO1', 'ESTADO2', 'Valor_Saldo', 'Saldo_Unid_Nuevo', 'NRO_DOCTO']
cols_presentes = [c for c in cols_requeridas if c in df.columns]
cols_faltantes = [c for c in cols_requeridas if c not in df.columns]

if cols_faltantes:
    print(f'[AVISO] Columnas no encontradas y seran omitidas: {cols_faltantes}')

# --- Filtro de quiebres de saldo ---
# Se considera quiebre cuando ESTADO1 o ESTADO2 indican condicion de quiebre
df_q = df.copy()

# Construir mascara de quiebre segun columnas disponibles
mascara_quiebre = pd.Series(False, index=df_q.index)

if 'ESTADO1' in df_q.columns:
    mascara_quiebre = mascara_quiebre | (df_q['ESTADO1'].astype(str).str.upper().str.contains('QUIEBRE|SIN SALDO|FALTANTE|NEGATIVO', na=False))

if 'ESTADO2' in df_q.columns:
    mascara_quiebre = mascara_quiebre | (df_q['ESTADO2'].astype(str).str.upper().str.contains('QUIEBRE|SIN SALDO|FALTANTE|NEGATIVO', na=False))

# Si ninguna mascara captura registros, usar Valor_Saldo o Saldo_Unid_Nuevo negativo como proxy
if mascara_quiebre.sum() == 0:
    print('[INFO] ESTADO1/ESTADO2 no identificaron quiebres por texto. Usando Valor_Saldo < 0 o Saldo_Unid_Nuevo < 0 como proxy.')
    if 'Valor_Saldo' in df_q.columns:
        mascara_quiebre = mascara_quiebre | (pd.to_numeric(df_q['Valor_Saldo'], errors='coerce').fillna(0) < 0)
    if 'Saldo_Unid_Nuevo' in df_q.columns:
        mascara_quiebre = mascara_quiebre | (pd.to_numeric(df_q['Saldo_Unid_Nuevo'], errors='coerce').fillna(0) < 0)

df_quiebres = df_q[mascara_quiebre].copy()

if df_quiebres.empty:
    print('[AVISO] No se encontraron registros de quiebre con los criterios definidos. Verificar valores en ESTADO1, ESTADO2, Valor_Saldo y Saldo_Unid_Nuevo.')
else:
    # --- Preparacion de columna de periodo mensual ---
    if 'FECHA' in df_quiebres.columns:
        df_quiebres['FECHA'] = pd.to_datetime(df_quiebres['FECHA'], errors='coerce')
        df_quiebres = df_quiebres.dropna(subset=['FECHA'])
        df_quiebres['MES_PERIODO'] = df_quiebres['FECHA'].dt.to_period('M')
    else:
        print('[ERROR] Columna FECHA no disponible. No es posible generar la evolucion temporal.')
        raise KeyError('FECHA requerida para esta seccion.')

    # --- Conversion numerica de columnas de magnitud ---
    if 'Valor_Saldo' in df_quiebres.columns:
        df_quiebres['Valor_Saldo'] = pd.to_numeric(df_quiebres['Valor_Saldo'], errors='coerce').fillna(0)

    if 'Saldo_Unid_Nuevo' in df_quiebres.columns:
        df_quiebres['Saldo_Unid_Nuevo'] = pd.to_numeric(df_quiebres['Saldo_Unid_Nuevo'], errors='coerce').fillna(0)

    # --- Agregacion mensual ---
    agg_dict = {}

[INFO] ESTADO1/ESTADO2 no identificaron quiebres por texto. Usando Valor_Saldo < 0 o Saldo_Unid_Nuevo < 0 como proxy.


## 5. Estacionalidad y Patrones de Quiebre por Mes y Anio

Esta seccion examina si los quiebres de stock siguen patrones recurrentes a lo largo del calendario, cruzando el mes del anio con el anio de registro entre 2021 y 2025. Identificar meses que concentran sistematicamente mayor frecuencia de quiebres permite a Ecovida / Alimentos Claudet anticipar periodos criticos, ajustar niveles de inventario con anticipacion y coordinar la logistica de reabastecimiento de forma preventiva en lugar de reactiva.

In [6]:
# Seccion 5: Estacionalidad y Patrones de Quiebre por Mes y Anio

req_cols = ['FECHA', 'ESTADO1', 'ESTADO2', 'Valor_Saldo', 'NRO_DOCTO']
cols_presentes = [c for c in req_cols if c in df.columns]
missing = [c for c in req_cols if c not in df.columns]
if missing:
    print(f'Columnas faltantes: {missing}')

if 'FECHA' in df.columns and 'ESTADO1' in df.columns and 'ESTADO2' in df.columns:

    df_q = df.copy()

    # Filtrar quiebres: ESTADO1 == 'Quiebre' o ESTADO2 == 'Quiebre'
    mask_quiebre = (
        df_q.get('ESTADO1', df_q.iloc[:, 0]).astype(str).str.strip().str.lower().eq('quiebre') |
        df_q.get('ESTADO2', df_q.iloc[:, 0]).astype(str).str.strip().str.lower().eq('quiebre')
    )
    df_q = df_q[mask_quiebre].copy()

    # Asegurar columna ANO y MES_NUM
    if 'ANO' not in df_q.columns:
        df_q['ANO'] = df_q['FECHA'].dt.year
    df_q['MES_NUM'] = df_q['FECHA'].dt.month

    # Filtrar rango 2021-2025
    df_q = df_q[df_q['ANO'].between(2021, 2025)]

    if df_q.empty:
        print('No se encontraron registros de quiebre en el rango 2021-2025.')
    else:
        # Metrica: contar documentos unicos o filas por mes-anio
        if 'NRO_DOCTO' in df_q.columns:
            pivot_data = (
                df_q.groupby(['ANO', 'MES_NUM'])['NRO_DOCTO']
                .count()
                .reset_index()
                .rename(columns={'NRO_DOCTO': 'Quiebres'})
            )
        else:
            pivot_data = (
                df_q.groupby(['ANO', 'MES_NUM'])
                .size()
                .reset_index()
                .rename(columns={0: 'Quiebres'})
            )

        # Crear pivot: filas = mes, columnas = anio
        pivot = pivot_data.pivot(index='MES_NUM', columns='ANO', values='Quiebres').fillna(0)

        # Reindexar para asegurar todos los meses del 1 al 12
        pivot = pivot.reindex(range(1, 13), fill_value=0)

        # Mapeo de nombres de mes en espanol
        nombres_mes = {
            1: 'Ene',
            2: 'Feb',
            3: 'Mar',
            4: 'Abr',
            5: 'May',
            6: 'Jun',
            7: 'Jul',
            8: 'Ago',
            9: 'Sep',
            10: 'Oct',
            11: 'Nov',
            12: 'Dic'
        }
        pivot.index = [nombres_mes.get(m, str(m)) for m in pivot.index]

No se encontraron registros de quiebre en el rango 2021-2025.


## 6. Sintesis Ejecutiva, Riesgos Priorizados y Recomendaciones

Esta seccion integra los hallazgos de las cuatro dimensiones de analisis (productos, clientes, canales y periodos) en una vision consolidada del riesgo economico y operacional para Ecovida / Alimentos Claudet. La combinacion de producto critico, cliente de alto valor y periodo de mayor frecuencia define el perfil de riesgo prioritario que concentra el mayor impacto sobre el resultado del negocio. A partir de dicho perfil se proponen acciones concretas y priorizadas para mitigar las vulnerabilidades identificadas.

In [7]:
# ============================================================
# SECCION 6: Sintesis Ejecutiva, Riesgos Priorizados y
#            Recomendaciones
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 0. Validacion de columnas disponibles
# ------------------------------------------------------------
COLS_REQUERIDAS = [
    'COD_ARTICULO', 'DESCRIPCION', 'NOM_CLIENTE',
    'Valor_Saldo', 'Saldo_Unid_Nuevo', 'ESTADO1', 'ESTADO2', 'FECHA'
]
cols_presentes = {c: (c in df.columns) for c in COLS_REQUERIDAS}

---

## Conclusiones

Este analisis respondio las siguientes preguntas de negocio:

- ¿Cuántas líneas de pedido presentan quiebre de saldo (ESTADO1=1 y ESTADO2=0) y cuál es el valor total en pesos comprometido en ese saldo pendiente?
- ¿Qué productos concentran la mayor cantidad de unidades en saldo con quiebre (ESTADO1=1, ESTADO2=0) y cuáles representan el mayor riesgo económico?
- ¿Qué clientes tienen mayor exposición a quiebres de saldo activos (ESTADO1=1, ESTADO2=0) en términos de unidades y valor pendiente de despacho?
- ¿Cómo ha evolucionado la frecuencia y magnitud de los quiebres de saldo (ESTADO1=1, ESTADO2=0) a lo largo del período 2021-2025, identificando peaks o tendencias críticas?

---
*Analisis generado con Python, pandas, matplotlib, seaborn*
*Dataset: 86,932 transacciones | 2021-03-23 hasta 2025-01-13*